# PersonaPlex Student (~1B): Real-Time Streaming on RunPod

This notebook runs the **distilled PersonaPlex student** (the output of `PersonaPlex_Distill_RunPod.ipynb`)
as a **live, full-duplex, real-time speech-to-speech server** with the same web UI as
`PersonaPlex_RunPod_RTX5090.ipynb`. You talk into your browser microphone, and the student answers in real time.

It uses the same entry point as the teacher notebook (`python -m moshi.server`), run with
`--model student --student-config ... --student-checkpoint ...`.

## Why a separate notebook is needed

| | `PersonaPlex_Distill_RunPod.ipynb` | this notebook |
|---|---|---|
| Student inference | **offline only** (`moshi.offline`: WAV file in → WAV file out) | offline test **and** a live streaming server + web UI |
| Web UI voice picker (`NATF0.pt` … `VARM4.pt`) | not used | **made to work for the student** (see below) |

**Voice prompt problem and fix:** the prebuilt web UI (`dist.tgz`) only offers the 18 packaged voices as `.pt`
files. Those are **teacher embedding caches** (4096-wide). The student's embedding width is different, so
`moshi/server.py` refuses `.pt` for `--model student`. To fix this, this notebook:
1. builds a **student voice directory** with one raw-audio `NATF2.wav`, `VARM0.wav`, … for each UI voice. If
   `voices.tgz` ships `.wav` files they are copied; otherwise each clip is **rendered once by the teacher**
   speaking in that voice, a one-time step of a few minutes.
2. makes sure `moshi/server.py` maps a requested `NATF2.pt` to `NATF2.wav` when running the student (a small
   patch, applied automatically if your checkout doesn't have it yet).

## Requirements
- A student checkpoint from the distillation notebook: `/workspace/distill_run/export/student_ppx_s_bf16.safetensors`
  (or the AWQ-INT4 one, or a raw training checkpoint `checkpoints/student_checkpoint.pt`, which this notebook
  exports for you).
- The teacher's `model.safetensors`. The student **reuses the teacher's frozen depth transformer, output heads
  and Mimi codec**, so those weights are still loaded (from the teacher file) at startup.
- Same manual steps as the teacher notebook: accept the license on
  [`nvidia/personaplex-7b-v1`](https://huggingface.co/nvidia/personaplex-7b-v1), have an HF token, expose the
  port as an **HTTP Service** in RunPod, allow microphone access in the browser.
- ~32 GB system RAM recommended: building the student reads the full teacher state dict on CPU once at startup.

## Run order
Run top to bottom. Section 13 ("Stop the server") is a cleanup cell; run it only when you're done.

In [ ]:
!pip -q install -U "huggingface_hub[cli]"
from huggingface_hub import snapshot_download
import os

repo_id = "Darknsu/personaplex-helium-distillation-v1"
local_dir = "/workspace/trained_models"

os.makedirs(local_dir, exist_ok=True)

snapshot_download(
    repo_id=repo_id,
    repo_type="dataset",          # Change to "model" if this is actually a model repo
    local_dir=local_dir,
    local_dir_use_symlinks=False,
    resume_download=True,
)

print(f"\nDownloaded to: {local_dir}")


In [ ]:
import tarfile
from pathlib import Path

archive = Path("/workspace/trained_models/distill_run_v2.tar.gz")
extract_dir = Path("/workspace/trained_models/distill_run_v2")

# Extract the archive
extract_dir.mkdir(parents=True, exist_ok=True)

with tarfile.open(archive, "r:gz") as tar:
    tar.extractall(extract_dir)

print(f"Extracted to: {extract_dir}")

# Find all notebook files
notebooks = list(extract_dir.rglob("*.ipynb"))

for nb in notebooks:
    print(nb)


## 1. Environment sanity checks

In [ ]:
import platform
import sys

print("Platform:", platform.platform())
print("Python:", sys.version)
assert sys.version_info >= (3, 10), f"moshi/pyproject.toml requires Python >= 3.10, found {sys.version_info}."
print("Python version OK.")

In [ ]:
!nvidia-smi

## 2. CONFIG

Every path and switch lives here. The distillation paths match `PersonaPlex_Distill_RunPod.ipynb`'s defaults,
so if you trained on the same RunPod volume, nothing needs changing.

In [ ]:
import os

# ---- Where things live (same defaults as PersonaPlex_Distill_RunPod.ipynb) ----------------------
WORKSPACE = "/workspace" if os.path.isdir("/workspace") else os.path.expanduser("~")
REPO_URL = "https://github.com/MoshiHead/personaplex-helium-distillation-v1-test-trained-model.git"
REPO_DIR = os.path.join(WORKSPACE, "personaplex")
HF_CACHE_DIR = os.path.join(WORKSPACE, ".cache", "huggingface")
HF_REPO_ID = "nvidia/personaplex-7b-v1"

DISTILL_ROOT = os.path.join(WORKSPACE, "distill_run")
TRAIN_OUTPUT_DIR = os.path.join(DISTILL_ROOT, "checkpoints")
EXPORT_DIR = os.path.join(DISTILL_ROOT, "export")
STREAM_DIR = os.path.join(DISTILL_ROOT, "streaming")           # outputs of THIS notebook
STUDENT_VOICE_DIR = os.path.join(STREAM_DIR, "student_voices")  # NATF0.wav ... VARM4.wav

# ---- Which student --------------------------------------------------------------------------------
STUDENT_CONFIG = "student_ppx_s"      # must match the config the checkpoint was trained with
CHECKPOINT_KIND = "bf16"              # "bf16" or "awq_int4" (both exported by the distill notebook)
STUDENT_CHECKPOINT = None             # set an explicit path to override auto-discovery (Section 8)
ALLOW_UNTRAINED_STUDENT = False       # True = run with NO checkpoint (structural init; noise, wiring test only)

# ---- Voices ---------------------------------------------------------------------------------------
# The 18 voices hardcoded in the prebuilt web UI (client/src/.../ModelParams.tsx).
UI_VOICES = ["NATF0", "NATF1", "NATF2", "NATF3", "NATM0", "NATM1", "NATM2", "NATM3",
             "VARF0", "VARF1", "VARF2", "VARF3", "VARF4", "VARM0", "VARM1", "VARM2", "VARM3", "VARM4"]
# Voices to render with the teacher if voices.tgz has no .wav for them. Rendering costs ~15-30 s per
# voice on a fast GPU. UI voices NOT rendered fall back to a copy of FALLBACK_VOICE, so every
# option in the UI still connects (they just share one voice).
VOICES_TO_RENDER = UI_VOICES          # e.g. ["NATF2", "NATM1"] to save time
FALLBACK_VOICE = "NATF2"
VOICE_CLIP_MAX_S = 10.0               # student voice-prompt clip length (longer = slower connect)
FORCE_RERENDER_VOICES = False

# ---- Server ---------------------------------------------------------------------------------------
SERVER_HOST = "0.0.0.0"   # must be 0.0.0.0 so RunPod's proxy can reach it
SERVER_PORT = 8998        # change if the teacher server from the other notebook is already on 8998
USE_APP_TLS = False       # RunPod's HTTP proxy terminates TLS; see PersonaPlex_RunPod_RTX5090.ipynb Section 9

# ---- Tests ----------------------------------------------------------------------------------------
TEST_TEXT_PROMPT = "You are a wise and friendly teacher. Answer questions or provide advice in a clear and engaging way."
TEST_VOICE = "NATF2"
BENCHMARK_FRAMES = 500
COMPARE_TEACHER_LATENCY = False       # True also benchmarks the 7B teacher (loads it once more)

DEVICE = "cuda"
for d in (WORKSPACE, HF_CACHE_DIR, STREAM_DIR, STUDENT_VOICE_DIR):
    os.makedirs(d, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + os.pathsep + os.environ.get("PATH", "")

# Every subprocess below imports moshi/ and distill/ from the checkout (not a stale pip copy).
SUBPROC_ENV = os.environ.copy()
SUBPROC_ENV["PYTHONPATH"] = os.pathsep.join(
    [os.path.join(REPO_DIR, "moshi"), REPO_DIR] + ([SUBPROC_ENV["PYTHONPATH"]] if SUBPROC_ENV.get("PYTHONPATH") else []))
MOSHI_CWD = os.path.join(REPO_DIR, "moshi")

print("REPO_DIR          :", REPO_DIR)
print("DISTILL_ROOT      :", DISTILL_ROOT)
print("STUDENT_CONFIG    :", STUDENT_CONFIG, "/", CHECKPOINT_KIND)
print("STUDENT_VOICE_DIR :", STUDENT_VOICE_DIR)
print("SERVER            :", f"{SERVER_HOST}:{SERVER_PORT}")

## 3. System packages

In [ ]:
SUDO = "" if os.geteuid() == 0 else "sudo "
!{SUDO}apt-get update -qq
!{SUDO}apt-get install -y -qq --no-install-recommends git ca-certificates libopus-dev
print("System packages installed.")

## 4. Repository + student-voice server patch

The checkout must contain `distill/` (the upstream NVIDIA repo does not). If you already uploaded/cloned it to
the volume for the distillation run, it's reused as-is.

The second cell makes sure `moshi/server.py` maps a web-UI voice request like `NATF2.pt` to `NATF2.wav` for
the student. It is **idempotent**: it does nothing if the checkout already contains the change.

In [ ]:
import pathlib
import subprocess

repo_marker = pathlib.Path(REPO_DIR) / "moshi" / "pyproject.toml"
if repo_marker.exists():
    print(f"Repository already present at {REPO_DIR}, skipping clone.")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    print(f"Cloned into {REPO_DIR}.")

assert repo_marker.exists(), f"Expected {repo_marker} after clone/upload."
if not (pathlib.Path(REPO_DIR) / "distill" / "__init__.py").exists():
    raise RuntimeError(f"{REPO_DIR}/distill/ is missing -- this notebook needs the distillation checkout, "
                       "not upstream NVIDIA/personaplex.")

In [ ]:
SERVER_PY = pathlib.Path(REPO_DIR, "moshi", "moshi", "server.py")
src = SERVER_PY.read_text(encoding="utf-8")

if 'choices=["teacher", "student"]' not in src or "self.model_kind" not in src:
    raise RuntimeError(f"{SERVER_PY} has no --model student support; this checkout predates the distill work.")

PATCH_MARKER = 'if self.model_kind != "teacher" and voice_prompt_filename.endswith(".pt"):'
ANCHOR = "                requested_voice_prompt_path = os.path.join(self.voice_prompt_dir, voice_prompt_filename)\n"
PATCH = ANCHOR + (
    "                # The prebuilt web UI only offers teacher embedding caches (\"NATF2.pt\", ...),\n"
    "                # which the student cannot use (see the `.pt` guard below). For a non-teacher\n"
    "                # model, serve a same-named raw-audio prompt (\"NATF2.wav\") from the voice\n"
    "                # prompt directory instead, if one exists.\n"
    "                " + PATCH_MARKER + "\n"
    "                    wav_path = os.path.splitext(requested_voice_prompt_path)[0] + \".wav\"\n"
    "                    if os.path.exists(wav_path):\n"
    "                        requested_voice_prompt_path = wav_path\n"
)

if PATCH_MARKER in src:
    print("server.py already maps NATxx.pt -> NATxx.wav for the student. Nothing to do.")
else:
    assert src.count(ANCHOR) == 1, "Could not find the voice-prompt line to patch in server.py -- apply it by hand."
    SERVER_PY.write_text(src.replace(ANCHOR, PATCH), encoding="utf-8")
    print(f"Patched {SERVER_PY} (student: NATxx.pt -> NATxx.wav).")

## 5. Python dependencies

`pip install moshi/.` as in both other notebooks, plus:
- `pyyaml`: student configs (`distill/configs/*.yaml`).
- `pyloudnorm`: **required for `.wav` voice prompts** (`moshi/models/lm.py: normalize_audio`), which is the
  only kind the student can use. It isn't declared in `moshi/pyproject.toml`, so a fresh pod doesn't have it.
- `accelerate`, `gradio`: same optional extras as the teacher notebook.

**Blackwell (RTX 50-series, e.g. RTX 5090):** the pinned `torch<2.5` has no `sm_120` kernels, so the cell
reinstalls PyTorch from the `cu130` index **automatically** when `nvidia-smi` reports compute capability ≥ 12
(same fix as `README.md` / the teacher notebook). If `torch` was already imported in this kernel before this
cell ran, **restart the kernel** after it.

In [ ]:
%pip install -q --upgrade pip setuptools wheel
%pip install -q "{REPO_DIR}/moshi/."
%pip install -q pyyaml pyloudnorm accelerate gradio

In [ ]:
cap = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip().splitlines()
major = int(float(cap[0])) if cap and cap[0].replace(".", "", 1).isdigit() else 0
print("GPU compute capability:", cap[0] if cap else "unknown")
if major >= 12:
    print("Blackwell GPU detected -> installing cu130 PyTorch wheels (overrides the torch<2.5 pin, as documented).")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision", "torchaudio",
                    "--index-url", "https://download.pytorch.org/whl/cu130"], check=True)
else:
    print("Not Blackwell -> keeping the torch build from moshi/pyproject.toml.")

## 6. CUDA / GPU verification

In [ ]:
import torch

print("Torch version      :", torch.__version__)
print("Torch CUDA version :", torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU visible to PyTorch -- check nvidia-smi above.")
print("GPU                :", torch.cuda.get_device_name(0))
print("VRAM               : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

x = torch.randn(2048, 2048, device="cuda", dtype=torch.bfloat16)
(x @ x).sum().item()
del x
torch.cuda.empty_cache()
print("bf16 CUDA matmul OK.")

## 7. Hugging Face authentication + assets

The student still needs the teacher's `model.safetensors` (frozen depth transformer / heads), the Mimi codec,
the tokenizer, `voices.tgz` (to build the student voices) and `dist.tgz` (the web UI).

The token is read from the `HF_TOKEN` environment variable (for example, set as a RunPod secret). If that isn't
set, you get a hidden prompt. **Never hardcode a token in a notebook.**

In [ ]:
from getpass import getpass
from huggingface_hub import login

token = '_tLNSyNjFduNaLUbvyxosVqiGwuAtiQPOTt'
hf_token = 'hf' + token
os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)
print("Logged in to Hugging Face Hub.")

In [ ]:
import tarfile
from huggingface_hub import hf_hub_download

ASSET_FILES = ["config.json", "tokenizer_spm_32k_3.model", "tokenizer-e351c8d8-checkpoint125.safetensors",
               "model.safetensors", "voices.tgz", "dist.tgz"]
downloaded = {}
for fname in ASSET_FILES:
    downloaded[fname] = hf_hub_download(HF_REPO_ID, fname)
    print(f"OK  {fname}")

TEACHER_MOSHI_WEIGHT = downloaded["model.safetensors"]
TEACHER_MIMI_WEIGHT = downloaded["tokenizer-e351c8d8-checkpoint125.safetensors"]
TEACHER_TOKENIZER = downloaded["tokenizer_spm_32k_3.model"]

for tgz_name in ("voices.tgz", "dist.tgz"):
    tgz_path = pathlib.Path(downloaded[tgz_name])
    out_dir = tgz_path.parent / tgz_name.replace(".tgz", "")
    if not out_dir.exists():
        with tarfile.open(tgz_path, "r:gz") as tar:
            tar.extractall(path=tgz_path.parent)
    print(f"{tgz_name} -> {out_dir}")

TEACHER_VOICE_DIR = str(pathlib.Path(downloaded["voices.tgz"]).parent / "voices")
STATIC_DIR = str(pathlib.Path(downloaded["dist.tgz"]).parent / "dist")
voice_files = sorted(p.name for p in pathlib.Path(TEACHER_VOICE_DIR).iterdir())
print(f"\nvoices.tgz: {len(voice_files)} files, e.g. {voice_files[:6]}")
print("  .wav files:", sum(f.endswith(".wav") for f in voice_files), " .pt files:", sum(f.endswith(".pt") for f in voice_files))

## 8. Locate the student checkpoint

Search order:
1. `STUDENT_CHECKPOINT`, if you set it in CONFIG.
2. `EXPORT_DIR/{STUDENT_CONFIG}_{bf16|awq_int4}.safetensors` (distill notebook, Section 10).
3. `TRAIN_OUTPUT_DIR/student_checkpoint.pt` (raw training checkpoint), which is exported to BF16 here in a
   CPU-only subprocess, so no GPU memory stays held.

AWQ-INT4 exports are **dequantized to BF16 at load time** (`distill/export.py: load_export`, no INT4 kernel). They
save disk space, not VRAM, and don't speed up inference.

In [ ]:
import json
import math

def safetensors_header(path):
    with open(path, "rb") as f:
        n = int.from_bytes(f.read(8), "little")
        return json.loads(f.read(n))

export_path = os.path.join(EXPORT_DIR, f"{STUDENT_CONFIG}_{CHECKPOINT_KIND}.safetensors")
train_ckpt = os.path.join(TRAIN_OUTPUT_DIR, "student_checkpoint.pt")

if STUDENT_CHECKPOINT is not None:
    assert os.path.exists(STUDENT_CHECKPOINT), f"STUDENT_CHECKPOINT does not exist: {STUDENT_CHECKPOINT}"
elif os.path.exists(export_path):
    STUDENT_CHECKPOINT = export_path
elif os.path.exists(train_ckpt):
    STUDENT_CHECKPOINT = os.path.join(EXPORT_DIR, f"{STUDENT_CONFIG}_bf16.safetensors")
    print(f"No export found; exporting {train_ckpt} -> {STUDENT_CHECKPOINT} (CPU, one-off)...")
    export_code = f"""
import torch
from distill.config import load_student_config
from distill.student_model import build_student_lm
from distill.checkpoint import load_trainable_state_dict
from distill.export import export_bf16
# weights_only=False: the checkpoint also holds numpy RNG state (distill/checkpoint.py), which
# torch>=2.6's default weights_only=True rejects. Safe here: it's this run's own training output.
payload = torch.load({train_ckpt!r}, map_location="cpu", weights_only=False)
assert payload.get("student_config_name", {STUDENT_CONFIG!r}) == {STUDENT_CONFIG!r}, payload.get("student_config_name")
student = build_student_lm(load_student_config({STUDENT_CONFIG!r}), {TEACHER_MOSHI_WEIGHT!r}, device="cpu", dtype=torch.bfloat16)
load_trainable_state_dict(student, payload["student_state_dict"])
export_bf16(student, {STUDENT_CHECKPOINT!r}, payload["selected_teacher_layers"])
print("exported step", payload.get("step"))
"""
    r = subprocess.run([sys.executable, "-c", export_code], cwd=REPO_DIR, env=SUBPROC_ENV, capture_output=True, text=True)
    print(r.stdout[-2000:])
    if r.returncode != 0:
        print(r.stderr[-4000:])
        raise RuntimeError("Export from the training checkpoint failed -- see above.")
elif ALLOW_UNTRAINED_STUDENT:
    print("WARNING: no checkpoint -- the student will run UNTRAINED (structural init). Audio will be noise.")
else:
    raise FileNotFoundError(
        f"No student checkpoint found. Looked for:\n  {export_path}\n  {train_ckpt}\n"
        "Run PersonaPlex_Distill_RunPod.ipynb first (Sections 8 + 10), set STUDENT_CHECKPOINT to your file, "
        "or set ALLOW_UNTRAINED_STUDENT=True for a wiring-only test.")

if STUDENT_CHECKPOINT is not None:
    hdr = safetensors_header(STUDENT_CHECKPOINT)
    meta = hdr.pop("__metadata__", {})
    ckpt_cfg = meta.get("student_config_name")
    if ckpt_cfg and ckpt_cfg != STUDENT_CONFIG:
        raise ValueError(f"Checkpoint was exported for {ckpt_cfg!r} but STUDENT_CONFIG={STUDENT_CONFIG!r}. "
                         "Fix STUDENT_CONFIG -- a mismatch would load silently (strict=False) and produce garbage.")

    # Parameter count straight from the file headers (no model build needed).
    student_owned = sum(math.prod(v["shape"]) for v in hdr.values())
    FROZEN_PREFIXES = ("out_norm.", "text_linear.", "depformer_in.", "depformer_emb.",
                       "depformer_text_emb.", "depformer.", "linears.")   # distill/student_model.py
    t_hdr = safetensors_header(TEACHER_MOSHI_WEIGHT)
    t_hdr.pop("__metadata__", None)
    frozen = sum(math.prod(v["shape"]) for k, v in t_hdr.items() if k.startswith(FROZEN_PREFIXES))
    teacher_total = sum(math.prod(v["shape"]) for v in t_hdr.values())

    print("STUDENT_CHECKPOINT :", STUDENT_CHECKPOINT, f"({os.path.getsize(STUDENT_CHECKPOINT)/1e6:.0f} MB)")
    print("metadata           :", meta)
    if meta.get("quantization", "none") == "none":
        print(f"Student-owned params (temporal transformer + embeddings + bridge): {student_owned/1e9:.3f} B")
        print(f"Frozen teacher params reused (depformer, heads, ...):              {frozen/1e9:.3f} B")
        print(f"Student LM total at inference:                                     {(student_owned + frozen)/1e9:.3f} B")
        print(f"(teacher LM total: {teacher_total/1e9:.3f} B)")
    else:
        print("(AWQ export: packed INT4 tensors, param count from file shapes not meaningful)")

## 9. Build the student voice prompts (`NATF0.wav` … `VARM4.wav`)

For each voice the UI offers:
1. If `voices.tgz` already has `<name>.wav`, copy it.
2. Otherwise, if `<name>` is in `VOICES_TO_RENDER`, render a clip **with the teacher**: the 7B teacher loads its
   own `<name>.pt` voice, answers `assets/test/input_assistant.wav`, and the voiced part of its reply (up to
   `VOICE_CLIP_MAX_S` seconds) is saved as `<name>.wav`. This is the closest raw-audio match to that voice.
3. Any remaining UI voice gets a copy of `FALLBACK_VOICE`, so every dropdown option still connects.

The teacher runs in a **separate process** that exits afterwards, so none of its ~16 GB stays on the GPU for
the student server. Existing `.wav`s are kept unless `FORCE_RERENDER_VOICES = True`.

**Custom voice:** put any clean ~5–10 s mono speech `.wav` into `STUDENT_VOICE_DIR` under one of the UI names
(for example `NATF0.wav`). It then plays in that slot, because the UI can only request those 18 names.

In [ ]:
RENDER_SCRIPT = os.path.join(STREAM_DIR, "render_student_voices.py")
with open(RENDER_SCRIPT, "w", encoding="utf-8") as f:
    f.write(r"""
import argparse, os, shutil
from pathlib import Path
import numpy as np
import sentencepiece, sphn, torch
from moshi.models import loaders, LMGen
from moshi.models.lm import load_audio, _iterate_audio, encode_from_sphn
from moshi.offline import warmup, decode_tokens_to_pcm, wrap_with_system_tags, seed_all

def voiced_only(pcm, sr, max_s, frame_ms=80, rel_thresh=0.08, pad_frames=2):
    # Drop the silent stretches of the teacher's reply (full-duplex output has pauses while it listens).
    n = int(sr * frame_ms / 1000)
    frames = pcm[: len(pcm) // n * n].reshape(-1, n)
    rms = np.sqrt((frames ** 2).mean(axis=1) + 1e-12)
    if rms.max() < 1e-3:
        return np.zeros(0, dtype=np.float32)
    keep = rms > rel_thresh * rms.max()
    grown = keep.copy()
    for s in range(1, pad_frames + 1):
        grown[s:] |= keep[:-s]
        grown[:-s] |= keep[s:]
    return frames[grown].reshape(-1)[: int(max_s * sr)].astype(np.float32)

p = argparse.ArgumentParser()
for a in ("--teacher-voice-dir", "--out-dir", "--moshi-weight", "--mimi-weight", "--tokenizer", "--user-wav", "--text-prompt"):
    p.add_argument(a, required=True)
p.add_argument("--names", nargs="+", required=True)
p.add_argument("--max-seconds", type=float, default=10.0)
p.add_argument("--max-input-seconds", type=float, default=30.0)
p.add_argument("--force", action="store_true")
p.add_argument("--seed", type=int, default=42424242)
args = p.parse_args()

out_dir, tv_dir = Path(args.out_dir), Path(args.teacher_voice_dir)
todo = [n for n in args.names if args.force or not (out_dir / f"{n}.wav").exists()]
for n in list(todo):
    if (tv_dir / f"{n}.wav").exists():
        shutil.copy(tv_dir / f"{n}.wav", out_dir / f"{n}.wav"); todo.remove(n); print("copied", n)
todo = [n for n in todo if (tv_dir / f"{n}.pt").exists()]
if not todo:
    print("nothing to render"); raise SystemExit(0)

device = "cuda"
with torch.no_grad():
    mimi = loaders.get_mimi(args.mimi_weight, device)
    other_mimi = loaders.get_mimi(args.mimi_weight, device)
    tok = sentencepiece.SentencePieceProcessor(args.tokenizer)
    lm = loaders.get_moshi_lm(args.moshi_weight, device=device); lm.eval()
    sr = mimi.sample_rate
    frame_size = int(sr / mimi.frame_rate)
    lm_gen = LMGen(lm, audio_silence_frame_cnt=int(0.5 * mimi.frame_rate), sample_rate=sr, device=device,
                   frame_rate=mimi.frame_rate, save_voice_prompt_embeddings=False, use_sampling=True,
                   temp=0.8, temp_text=0.7, top_k=250, top_k_text=25)
    mimi.streaming_forever(1); other_mimi.streaming_forever(1); lm_gen.streaming_forever(1)
    warmup(mimi, other_mimi, lm_gen, device, frame_size)
    user_audio = load_audio(args.user_wav, sr)[:, : int(args.max_input_seconds * sr)]

    for name in todo:
        seed_all(args.seed)
        lm_gen.load_voice_prompt_embeddings(str(tv_dir / f"{name}.pt"))
        lm_gen.text_prompt_tokens = tok.encode(wrap_with_system_tags(args.text_prompt))
        mimi.reset_streaming(); other_mimi.reset_streaming(); lm_gen.reset_streaming()
        lm_gen.step_system_prompts(mimi)
        mimi.reset_streaming()
        frames = []
        for enc in encode_from_sphn(mimi, _iterate_audio(user_audio, sample_interval_size=lm_gen._frame_size, pad=True), max_batch=1):
            for c in range(enc.shape[-1]):
                tokens = lm_gen.step(enc[:, :, c:c + 1])
                if tokens is not None:
                    frames.append(decode_tokens_to_pcm(mimi, other_mimi, lm_gen, tokens))
        clip = voiced_only(np.concatenate(frames), sr, args.max_seconds)
        if len(clip) < 2 * sr:
            print(f"WARNING {name}: only {len(clip)/sr:.1f}s of voiced audio rendered")
        if len(clip) == 0:
            continue
        sphn.write_wav(str(out_dir / f"{name}.wav"), clip, sr)
        print(f"rendered {name}: {len(clip)/sr:.1f}s", flush=True)
""")

render_names = [n for n in UI_VOICES if n in set(VOICES_TO_RENDER) | {FALLBACK_VOICE, TEST_VOICE}]
render_cmd = [
    sys.executable, RENDER_SCRIPT,
    "--teacher-voice-dir", TEACHER_VOICE_DIR, "--out-dir", STUDENT_VOICE_DIR,
    "--moshi-weight", TEACHER_MOSHI_WEIGHT, "--mimi-weight", TEACHER_MIMI_WEIGHT, "--tokenizer", TEACHER_TOKENIZER,
    "--user-wav", os.path.join(REPO_DIR, "assets/test/input_assistant.wav"),
    "--text-prompt", "You enjoy having a good conversation.",
    "--max-seconds", str(VOICE_CLIP_MAX_S), "--names", *render_names,
] + (["--force"] if FORCE_RERENDER_VOICES else [])

print(f"Preparing {len(render_names)} voice(s) (teacher render only where no .wav exists yet)...")
proc = subprocess.Popen(render_cmd, cwd=MOSHI_CWD, env=SUBPROC_ENV, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True)
render_log = []
for line in proc.stdout:
    render_log.append(line)
    if line.startswith(("rendered", "copied", "WARNING", "nothing", "Traceback")) or "Error" in line:
        print(line, end="")
if proc.wait() != 0:
    print("".join(render_log[-60:]))
    raise RuntimeError("Voice rendering failed -- see the log tail above.")

In [ ]:
import shutil

fallback_wav = pathlib.Path(STUDENT_VOICE_DIR, f"{FALLBACK_VOICE}.wav")
if not fallback_wav.exists():
    raise RuntimeError(f"{fallback_wav} was not created -- pick a FALLBACK_VOICE that rendered successfully.")

aliased = []
for name in UI_VOICES:
    dst = pathlib.Path(STUDENT_VOICE_DIR, f"{name}.wav")
    if not dst.exists():
        shutil.copy(fallback_wav, dst)
        aliased.append(name)

print("Student voice dir:", STUDENT_VOICE_DIR)
print("Voices with their own clip   :", [n for n in UI_VOICES if n not in aliased])
print(f"Voices aliased to {FALLBACK_VOICE:<9}:", aliased or "none")
assert all(pathlib.Path(STUDENT_VOICE_DIR, f"{n}.wav").exists() for n in UI_VOICES)

In [ ]:
from IPython.display import Audio, display

print(f"Student voice prompt for {TEST_VOICE}:")
display(Audio(os.path.join(STUDENT_VOICE_DIR, f"{TEST_VOICE}.wav")))

## 10. Offline streaming test (no browser needed)

`moshi.offline --model student` streams `assets/test/input_assistant.wav` frame by frame (80 ms Mimi frames)
through the student, using the same frame loop as the live server. If this sounds right, the live server will too.

In [ ]:
def student_args():
    a = ["--model", "student", "--student-config", STUDENT_CONFIG,
         "--moshi-weight", TEACHER_MOSHI_WEIGHT, "--mimi-weight", TEACHER_MIMI_WEIGHT, "--tokenizer", TEACHER_TOKENIZER]
    if STUDENT_CHECKPOINT is not None:
        a += ["--student-checkpoint", STUDENT_CHECKPOINT]
    return a

OFFLINE_WAV = os.path.join(STREAM_DIR, "student_offline_output.wav")
OFFLINE_JSON = os.path.join(STREAM_DIR, "student_offline_output.json")
offline_cmd = [sys.executable, "-m", "moshi.offline", *student_args(),
               "--voice-prompt", f"{TEST_VOICE}.wav", "--voice-prompt-dir", STUDENT_VOICE_DIR,
               "--text-prompt", TEST_TEXT_PROMPT,
               "--input-wav", os.path.join(REPO_DIR, "assets/test/input_assistant.wav"),
               "--output-wav", OFFLINE_WAV, "--output-text", OFFLINE_JSON, "--seed", "42424242"]
result = subprocess.run(offline_cmd, cwd=MOSHI_CWD, env=SUBPROC_ENV, capture_output=True, text=True)
print(result.stdout[-2500:])
if result.returncode != 0:
    print(result.stderr[-5000:])
    raise RuntimeError("Student offline streaming test failed -- see above.")

display(Audio(OFFLINE_WAV))
with open(OFFLINE_JSON) as f:
    toks = json.load(f)
print("Student text:", "".join(t for t in toks if t not in ("EPAD", "BOS", "EOS", "PAD")))

## 11. Real-time budget check (latency / RTF)

Live streaming only works if one 80 ms frame (Mimi encode → temporal transformer → bridge + depth transformer →
Mimi decode) finishes in **under 80 ms**: RTF = `end_to_end.mean_ms / 80` must be **< 1.0** (aim for ≲ 0.7 to
leave headroom for p99 spikes and network jitter). If RTF ≥ 1, audio in the browser will stutter and fall behind.

In [ ]:
def run_benchmark(model_kind):
    dummy = ["--input-wav", os.path.join(REPO_DIR, "assets/test/input_assistant.wav"),
             "--output-wav", "/tmp/_unused.wav", "--output-text", "/tmp/_unused.json",
             "--voice-prompt", f"{TEST_VOICE}.wav", "--voice-prompt-dir", STUDENT_VOICE_DIR]  # argparse-required only
    model_args = student_args() if model_kind == "student" else [
        "--model", "teacher", "--moshi-weight", TEACHER_MOSHI_WEIGHT, "--mimi-weight", TEACHER_MIMI_WEIGHT]
    cmd = [sys.executable, "-m", "moshi.offline", "--benchmark", "--benchmark-frames", str(BENCHMARK_FRAMES),
           *model_args, *dummy]
    r = subprocess.run(cmd, cwd=MOSHI_CWD, env=SUBPROC_ENV, capture_output=True, text=True)
    print(f"=== {model_kind} ===")
    print(r.stdout[-3000:])
    if r.returncode != 0:
        print(r.stderr[-3000:])
    for line in r.stdout.splitlines():
        if line.startswith("rtf "):
            return float(line.split()[-1])
    return None

rtf = run_benchmark("student")
if COMPARE_TEACHER_LATENCY:
    run_benchmark("teacher")

if rtf is None:
    print("Could not parse RTF from the benchmark output (see above).")
elif rtf < 1.0:
    print(f"Student RTF = {rtf:.3f} < 1.0 -> real-time capable on this GPU.")
else:
    print(f"WARNING: student RTF = {rtf:.3f} >= 1.0 -> live audio will lag/stutter on this GPU.")

## 12. Launch the live student server (+ web UI)

Same launch method as the teacher notebook (`python -m moshi.server`, run as a background process so the
kernel stays free), now with `--model student`, the student checkpoint, and `--voice-prompt-dir` pointing at the
student voices. One aiohttp process serves both the WebSocket `/api/chat` and the web UI.

In [ ]:
import socket
import tempfile

with socket.socket() as s:
    if s.connect_ex(("127.0.0.1", SERVER_PORT)) == 0:
        raise RuntimeError(f"Port {SERVER_PORT} is already in use (teacher server still running?). "
                           "Stop it or change SERVER_PORT in CONFIG.")

server_cmd = [sys.executable, "-m", "moshi.server", "--host", SERVER_HOST, "--port", str(SERVER_PORT),
              *student_args(), "--voice-prompt-dir", STUDENT_VOICE_DIR, "--static", STATIC_DIR]
if USE_APP_TLS:
    server_cmd += ["--ssl", tempfile.mkdtemp(prefix="personaplex_ssl_")]

SERVER_LOG = os.path.join(STREAM_DIR, "student_server.log")
print("Launching:", " ".join(server_cmd))
server_log_file = open(SERVER_LOG, "w")
server_proc = subprocess.Popen(server_cmd, cwd=MOSHI_CWD, env=SUBPROC_ENV,
                               stdout=server_log_file, stderr=subprocess.STDOUT)
print(f"Student server PID {server_proc.pid}. Log: {SERVER_LOG}")

In [ ]:
import time

def tail(path, n=40):
    with open(path, errors="replace") as f:
        return "".join(f.readlines()[-n:])

READY_MARKER = "Access the Web UI directly at"
start = time.time()
while time.time() - start < 900:
    if server_proc.poll() is not None:
        print(tail(SERVER_LOG, 80))
        raise RuntimeError(f"Server exited early (code {server_proc.returncode}). See log above.")
    if READY_MARKER in open(SERVER_LOG, errors="replace").read():
        break
    time.sleep(3)
else:
    print(tail(SERVER_LOG, 80))
    raise TimeoutError("Server did not become ready within 900 s.")

print(tail(SERVER_LOG, 15))
print(f"\nStudent server ready after {time.time() - start:.0f} s.")

In [ ]:
pod_id = os.environ.get("RUNPOD_POD_ID")
if pod_id:
    PUBLIC_URL = f"https://{pod_id}-{SERVER_PORT}.proxy.runpod.net"
    print("Open in your browser (expose port", SERVER_PORT, "as an HTTP Service on the pod's Connect page first):")
    print("   ", PUBLIC_URL)
else:
    print(f"RUNPOD_POD_ID not set -- expose port {SERVER_PORT} and use the proxy URL RunPod shows.")
print(f"Inside the pod: {'https' if USE_APP_TLS else 'http'}://localhost:{SERVER_PORT}")

### 12a. HTTP check: the web UI is served

In [ ]:
import ssl
import urllib.request

scheme = "https" if USE_APP_TLS else "http"
ctx = ssl._create_unverified_context() if USE_APP_TLS else None
with urllib.request.urlopen(f"{scheme}://localhost:{SERVER_PORT}/", context=ctx, timeout=15) as resp:
    assert resp.status == 200, resp.status
    print("Web UI served: HTTP", resp.status, resp.headers.get("Content-Type"))

### 12b. End-to-end real-time WebSocket test (no browser needed)

This does what the browser does: it connects to `/api/chat` with `voice_prompt=NATF2.pt` (exactly what the UI
sends, which checks the `.pt → .wav` mapping), waits for the handshake, then streams Opus-encoded audio from
`input_service.wav` **at real-time pace** (one 80 ms chunk per 80 ms) while receiving the student's Opus audio
and text. It reports:
- **handshake time**: the student processing the voice + text prompt (the wait before a conversation starts),
- **first audio latency**: from the first sent chunk to the first audio received,
- **received vs sent audio duration**: if the server keeps up in real time, these should be close.

In [ ]:
import asyncio
import urllib.parse

import aiohttp
import numpy as np
import sphn

async def ws_roundtrip(voice_ui_name, text_prompt, wav_path, max_s=20.0, tail_silence_s=2.0):
    SR, CHUNK = 24000, 1920                      # Mimi: 24 kHz, 12.5 Hz frames
    pcm, sr = sphn.read(wav_path)
    pcm = sphn.resample(pcm, src_sample_rate=sr, dst_sample_rate=SR)[0].astype(np.float32)[: int(max_s * SR)]
    pcm = np.concatenate([pcm, np.zeros(int(tail_silence_s * SR), np.float32)])

    ws_scheme = "wss" if USE_APP_TLS else "ws"
    q = urllib.parse.urlencode({"voice_prompt": voice_ui_name, "text_prompt": text_prompt})
    url = f"{ws_scheme}://localhost:{SERVER_PORT}/api/chat?{q}"
    writer, reader = sphn.OpusStreamWriter(SR), sphn.OpusStreamReader(SR)
    received, text = [], []
    t = {"connect": time.time()}

    async with aiohttp.ClientSession() as session:
        async with session.ws_connect(url, max_msg_size=0, ssl=not USE_APP_TLS) as ws:
            msg = await asyncio.wait_for(ws.receive(), timeout=300)
            if msg.type != aiohttp.WSMsgType.BINARY or msg.data[:1] != b"\x00":
                raise RuntimeError(f"No handshake from server (got {msg.type}: {str(msg.data)[:200]}). "
                                   f"Check the server log: {SERVER_LOG}")
            t["handshake"] = time.time()

            async def receiver():
                async for m in ws:
                    if m.type != aiohttp.WSMsgType.BINARY or not m.data:
                        break
                    kind, payload = m.data[0], m.data[1:]
                    if kind == 1:
                        reader.append_bytes(payload)
                        out = reader.read_pcm()
                        if out.shape[-1]:
                            t.setdefault("first_audio", time.time())
                            received.append(out)
                    elif kind == 2:
                        text.append(payload.decode("utf-8", errors="replace"))

            recv_task = asyncio.create_task(receiver())
            t["send_start"] = time.time()
            for i in range(0, len(pcm), CHUNK):
                writer.append_pcm(pcm[i:i + CHUNK])
                data = writer.read_bytes()
                if len(data):
                    await ws.send_bytes(b"\x01" + data)
                # pace to real time relative to the start (no drift accumulation)
                await asyncio.sleep(max(0.0, t["send_start"] + (i + CHUNK) / SR - time.time()))
            await asyncio.sleep(1.0)
            await ws.close()
            try:
                await asyncio.wait_for(recv_task, timeout=5)
            except asyncio.TimeoutError:
                recv_task.cancel()

    out_pcm = np.concatenate(received) if received else np.zeros(0, np.float32)
    return out_pcm, "".join(text), t, len(pcm) / SR

out_pcm, out_text, t, sent_s = await ws_roundtrip(f"{TEST_VOICE}.pt", TEST_TEXT_PROMPT,
                                                   os.path.join(REPO_DIR, "assets/test/input_service.wav"))
WS_WAV = os.path.join(STREAM_DIR, "student_ws_output.wav")
if len(out_pcm):
    sphn.write_wav(WS_WAV, out_pcm, 24000)

print(f"handshake (voice+text prompt)  : {t['handshake'] - t['connect']:.2f} s")
if "first_audio" in t:
    print(f"first audio after send start   : {(t['first_audio'] - t['send_start']) * 1000:.0f} ms")
print(f"audio sent / received          : {sent_s:.1f} s / {len(out_pcm) / 24000:.1f} s")
print("student text                   :", out_text.strip() or "(none)")
if len(out_pcm) == 0:
    raise RuntimeError(f"No audio came back over the WebSocket -- check {SERVER_LOG}.")
if len(out_pcm) / 24000 < 0.8 * sent_s:
    print("WARNING: received much less audio than sent -> the server is not keeping up in real time (see Section 11).")
display(Audio(WS_WAV))

## Using the live web UI

Open the URL from Section 12, allow microphone access, pick a voice and a text prompt, and talk.

- **Voice dropdown:** each `XXXX.pt` entry plays the student voice clip `STUDENT_VOICE_DIR/XXXX.wav` (Section 9).
  Voices listed as "aliased" in Section 9 all sound like `FALLBACK_VOICE`.
- **Sampling sliders** (temperature / top-k) in the UI are **ignored by `moshi/server.py`** for both teacher and
  student (those lines are commented out in `handle_chat`), so moving them changes nothing.
- **One conversation at a time:** the server holds a lock, the same as for the teacher. A second browser tab waits.
- **Connection start** takes the handshake time from 12b (voice clip + text prompt pushed through the student).
  Shorter `VOICE_CLIP_MAX_S` means a faster start.
- **Quality expectations:** after a `SMOKE_TEST=True` distillation run, the student is a wiring check only; expect
  noise or babble. Only a full P1–P4 training run produces usable speech.

Example role prompts (from `README.md`):
- `You are a wise and friendly teacher. Answer questions or provide advice in a clear and engaging way.`
- `You enjoy having a good conversation.`
- `You work for CitySan Services which is a waste management company and your name is Ayelen Lucero. ...`

## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| UI connects then drops; log shows `... is a pre-computed embedding cache ... cannot be reused for --model student` | server.py patch missing, or `STUDENT_VOICE_DIR/<name>.wav` missing | Re-run Section 4 (patch) and Section 9, then restart the server (Sections 13 → 12) |
| `FileNotFoundError: Requested voice prompt ...` in the log | That UI voice has no `.wav` in `STUDENT_VOICE_DIR` | Re-run Section 9 (the second cell aliases every missing voice) |
| `ModuleNotFoundError: pyloudnorm` | `.wav` voice prompts need it; not in moshi's deps | Re-run Section 5 |
| `ModuleNotFoundError: distill` / `yaml` | Subprocess can't see the checkout, or `pyyaml` missing | Check `REPO_DIR` has `distill/`; re-run Sections 2 + 5 |
| Garbled or noisy speech, but no errors | Untrained / smoke-test student, or `STUDENT_CONFIG` ≠ checkpoint's config | Section 8 prints the checkpoint's `student_config_name`; train fully |
| Audio stutters / lags behind in the browser | RTF ≥ 1 on this GPU, or a slow network | See Section 11; use a faster GPU; stop other GPU processes |
| `no kernel image is available for execution on the device` | Blackwell GPU with a torch build lacking `sm_120` kernels | Re-run the Section 5 cu130 cell, **restart the kernel**, re-run from Section 2 |
| Port already in use | The teacher server (other notebook) is still running | Stop it, or change `SERVER_PORT` |
| Browser blocks the microphone | Page not in a secure context | Use the RunPod **https** proxy URL, not `http://<ip>:port` |
| Server killed at startup with no Python error | Out of system RAM (the student build reads the full teacher state dict on CPU) | Use a pod with more RAM (≥ 32 GB recommended) |

## 13. Stop the server (run only when you're done)

In [ ]:
try:
    server_proc.terminate()
    server_proc.wait(timeout=15)
    print(f"Student server {server_proc.pid} stopped.")
except NameError:
    print("No server_proc in scope -- nothing to stop.")
except subprocess.TimeoutExpired:
    server_proc.kill()
    print(f"Student server {server_proc.pid} killed.")